# [LEC] Langfuse 기반 Observability & Evaluation

슬라이드와 설명, 퀴즈로 LLM 애플리케이션의 실행을 기록하고(Observability) 답변 품질에 점수를 남기는(Evaluation) 방법을 익힙니다.

**목차**

- 1. Langfuse (슬라이드) (퀴즈)
  - 1.1 Langfuse란 (슬라이드) (퀴즈)
  - 1.2 왜 실행을 추적하나 (슬라이드) (퀴즈)
  - 1.3 Trace (슬라이드) (퀴즈)
  - 1.4 CallbackHandler (슬라이드) (퀴즈)
  - 1.5 @observe (슬라이드) (퀴즈)
- 2. Evaluation (슬라이드) (퀴즈)
  - 2.1 Evaluation이란 (슬라이드) (퀴즈)
  - 2.2 Rule-based Evaluation (슬라이드) (퀴즈)
  - 2.3 LLM-as-a-Judge (슬라이드) (퀴즈)
- 3. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("4-2", [1, 2])

이번 시간은 두 가지를 봅니다.

1. Langfuse: 실행 기록(Trace)을 남기는 도구와, 그 기록을 붙이는 두 방법(CallbackHandler, @observe)
2. Evaluation: 답변 품질에 점수를 남기는 두 방법(Rule-based Evaluation, LLM-as-a-Judge)

## 1. Langfuse <small>(슬라이드) (퀴즈)</small>

### 1.1 Langfuse란 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [3, 4])

Langfuse는 LLM 애플리케이션을 관찰하고 평가하기 위한 오픈소스 Observability 플랫폼입니다.
프롬프트, 모델 호출, 도구 호출, 지연시간, 비용, 오류를 Trace 단위로 기록합니다.
LangChain, LangGraph, OpenAI SDK, Python SDK와 연동해 실행 흐름을 추적할 수 있고, 기록된 Trace로 디버깅, 비용 분석, 사용자 피드백, 품질 평가를 합니다.
클라우드로 쓸 수도 있고, Docker Compose로 자체 서버에 설치할 수도 있습니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q11.json")

**정리:** Langfuse는 LLM 애플리케이션의 실행을 Trace로 기록하고, 그 기록으로 디버깅·비용 분석·품질 평가를 하는 도구입니다.

### 1.2 왜 실행을 추적하나 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [5])

LLM 애플리케이션은 같은 입력처럼 보여도 내부 실행 경로가 달라질 수 있습니다. 그래서 다음을 확인해야 합니다.

- 어떤 프롬프트가 실제로 모델에 전달되었는지
- 어떤 도구가 어떤 입력으로 호출되었는지
- 어느 단계에서 지연시간과 비용이 생겼는지
- 실패한 응답이 모델 문제인지, 컨텍스트 문제인지, 도구 문제인지

Trace는 "무엇이 잘못됐는가"를 감으로 추측하지 않고 실행 기록으로 확인하게 합니다.

**왜 배우나:** Agent는 모델 호출, 도구 호출, 노드 실행이 여러 번 이어진 결과를 돌려줍니다. 최종 답변만 출력해서는 답이 틀렸을 때 어느 단계 탓인지 가를 수 없습니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q12.json")

**정리:** 같은 입력이라도 내부 경로가 다를 수 있습니다. Trace로 실제 프롬프트, 도구 호출, 지연·비용, 실패 원인을 확인합니다.

### 1.3 Trace <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [6])

Trace는 LLM 애플리케이션의 실행 과정을 단계별로 기록한 데이터입니다.
프롬프트 입력, 모델 호출, 도구 호출, LangGraph 노드 실행, 그리고 각 단계의 입력·출력·지연시간·오류가 담깁니다.
Trace를 보면 사용자 요청 하나가 내부에서 어떤 흐름으로 처리되었는지 다시 그려 볼 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q13.json")

**정리:** Trace = 요청 하나의 실행 과정을 단계별(입력·출력·지연시간·오류)로 남긴 기록입니다.

### 1.4 CallbackHandler <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [7])

CallbackHandler는 LangChain과 LangGraph의 실행 이벤트를 Langfuse Trace로 보내는 연결 지점입니다.
Chain, Runnable, Tool, LLM 호출 이벤트를 자동으로 모읍니다.
붙이는 방법은 두 가지입니다. 실행할 때 `config`의 `callbacks`로 넘기거나, `with_config`로 기본 설정을 만들어 두면 이후 실행마다 적용됩니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q14.json")

**정리:** CallbackHandler를 `config`의 `callbacks`로 넘기거나 `with_config`로 붙이면 LangChain·LangGraph 실행이 Trace로 기록됩니다.

### 1.5 @observe <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [8])

`@observe()`는 일반 Python 함수를 Langfuse Trace 또는 Span으로 기록하는 데코레이터입니다.
LangChain 밖에서 실행되는 사용자 정의 함수, 예를 들어 검색, 후처리, 평가, 파일 생성 같은 로직을 Trace에 넣을 수 있습니다.
그러면 요청 하나 안에서 LLM 호출과 일반 Python 처리를 함께 볼 수 있습니다.

**왜 배우나:** CallbackHandler는 LangChain·LangGraph의 실행만 모읍니다. RAG의 검색처럼 체인 밖에 있는 단계는 @observe를 붙여야 같은 기록 안에 나타납니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q15.json")

**정리:** 체인·그래프 실행은 CallbackHandler로, 체인 밖의 일반 함수는 @observe로 기록합니다.

## 2. Evaluation <small>(슬라이드) (퀴즈)</small>

### 2.1 Evaluation이란 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [9, 10])

Evaluation은 응답이 좋은지 나쁜지를 점수로 남기는 과정입니다.
트레이싱만으로는 답변 품질을 알 수 없습니다. Score를 남기면 품질 추세를 숫자로 볼 수 있고, 규칙 기반 평가와 LLM 평가를 함께 쓸 수 있습니다.

Observability가 실행 기록을 남긴다면, Evaluation은 그 실행 결과의 품질을 수치와 근거로 남깁니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q21.json")

**정리:** Trace는 무슨 일이 있었는지, Score는 그 결과가 얼마나 좋았는지를 남깁니다.

### 2.2 Rule-based Evaluation <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [11])

Rule-based Evaluation은 코드로 명확히 판단할 수 있는 조건을 평가합니다. 예를 들어 예시 포함 여부, 특정 키워드 포함 여부, 답변 길이입니다.
단순하지만 안정적입니다. 빠르고 저렴하며, 같은 답에는 늘 같은 점수가 나와 재현성이 높습니다.
다만 설명이 얼마나 쉬운지 같은 정성적 품질은 판단하기 어렵습니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q22.json")

**정리:** 규칙 기반 평가는 빠르고 싸고 재현성이 높지만, 정성적 품질은 판단하기 어렵습니다.

### 2.3 LLM-as-a-Judge <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-2", [12, 13])

LLM-as-a-Judge는 별도 LLM이 응답 품질을 평가하는 방식입니다.
답변의 유용성, 질문과의 관련성, 근거 충실성, 설명의 완성도처럼 규칙으로 판단하기 어려운 정성적 품질에 씁니다.

쓸 때는 평가 기준을 분명히 적어야 합니다.

- 평가 기준을 구체적으로 씁니다.
- 점수 범위를 고정해 근거 없는 높은 점수를 막습니다.
- 가능하면 규칙 기반 평가와 함께 씁니다.

**왜 배우나:** 캡스톤에서도 최종 보고서를 LLM Judge로 평가하고 점수를 Langfuse에 기록합니다. 평가 기준을 어떻게 쓰느냐에 따라 점수가 달라집니다.

In [ ]:
show_quiz("../assets/quizzes/4-2/q23.json")

**정리:** LLM Judge는 정성적 품질을 평가합니다. 기준을 구체적으로, 점수 범위를 고정해서, 가능하면 규칙 기반과 함께 씁니다.

## 3. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/4-2/final.json")

**이 노트북에서 본 것**
- Langfuse는 프롬프트, 모델·도구 호출, 지연시간, 비용, 오류를 Trace 단위로 기록합니다.
- Trace가 있으면 실패 원인을 감이 아니라 실행 기록으로 확인할 수 있습니다.
- LangChain·LangGraph 실행은 CallbackHandler를 `callbacks`로 넘기거나 `with_config`로 붙여 기록합니다.
- 체인 밖의 일반 함수는 @observe로 기록합니다.
- Evaluation은 결과 품질을 점수로 남깁니다. 규칙 기반 평가는 빠르고 재현성이 높고, LLM-as-a-Judge는 정성적 품질을 봅니다.
- LLM Judge는 기준을 구체적으로 쓰고 점수 범위를 고정하며, 가능하면 규칙 기반과 함께 씁니다.